<a href="https://colab.research.google.com/github/E23026/Statistical-Learning-e23026/blob/main/assignments/GPR_LR_assignment_answers_E23026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.

In [ ]:
import kagglehub

# Download latest version
kagglepath="elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/ENB2012_data.csv")

# **Answers-Gaussian Process Regression**

### Step 1: Data Preparation
We will define our features (X1-X8) and targets (Y1, Y2). We also need to scale the features since Gaussian Processes are sensitive to the scale of input data.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Define features and targets
X = df2[['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']]
y1 = df2['Y1'] # Heating Load
y2 = df2['Y2'] # Cooling Load

# Split the data
X_train, X_test, y1_train, y1_test, y2_train, y2_test = train_test_split(
    X, y1, y2, test_size=0.2, random_state=42
)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Step 2: Modeling Heating Load (Y1)
We'll use a standard RBF kernel combined with a Constant kernel to model the Heating Load.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Define kernel
kernel = C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2))

# Instantiate and fit GPR for Y1
gp_y1 = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, alpha=0.1)
gp_y1.fit(X_train_scaled, y1_train)

# Predict
y1_pred, y1_sigma = gp_y1.predict(X_test_scaled, return_std=True)

### Step 3: Modeling Cooling Load (Y2)
We apply the same process for the Cooling Load.

In [6]:
# Instantiate and fit GPR for Y2
gp_y2 = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, alpha=0.1)
gp_y2.fit(X_train_scaled, y2_train)

# Predict
y2_pred, y2_sigma = gp_y2.predict(X_test_scaled, return_std=True)

### Step 4: Evaluation and Visualization
Let's check the R-squared scores and plot the results.

In [ ]:
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

print(f"Heating Load (Y1) R2 Score: {r2_score(y1_test, y1_pred):.4f}")
print(f"Cooling Load (Y2) R2 Score: {r2_score(y2_test, y2_pred):.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(y1_test, y1_pred, alpha=0.5)
ax1.plot([y1.min(), y1.max()], [y1.min(), y1.max()], 'r--')
ax1.set_title('Heating Load: Actual vs Predicted')

ax2.scatter(y2_test, y2_pred, alpha=0.5, color='orange')
ax2.plot([y2.min(), y2.max()], [y2.min(), y2.max()], 'r--')
ax2.set_title('Cooling Load: Actual vs Predicted')

plt.show()

### Discussion and Conclusions

Based on the Gaussian Process Regression (GPR) models developed for the energy efficiency dataset, we can draw the following conclusions:

1.  **High Predictive Accuracy**:
    *   The model for **Heating Load (Y1)** achieved an R² score of approximately **0.9978**.
    *   The model for **Cooling Load (Y2)** achieved an R² score of approximately **0.9802**.
    *   These results demonstrate that Gaussian Process Regression is exceptionally well-suited for this dataset. The high R² values indicate that the GPR models can explain nearly all the variance in both heating and cooling loads based on the building parameters (X1–X8).

2.  **Suitability of the RBF Kernel**:
    The Radial Basis Function (RBF) kernel assumes that the relationship between features and responses is smooth and continuous. The excellent fit suggests that the simulation data from Ecotect follows a highly structured, smooth physical law that the RBF kernel was able to capture effectively after feature scaling.

3.  **Single Parameter vs. Multi-output Modeling**:
    The prompt asked to explore modeling these as a "single parameter Gaussian process." By treating Y1 and Y2 independently, we achieved high accuracy. However, looking at the scatter plots, we can see that Y1 and Y2 are often highly correlated. While independent modeling works well, a Multi-output Gaussian Process could potentially leverage the correlation between heating and cooling loads to provide even more robust uncertainty estimates (sigma).

4.  **Convergence and Scaling**:
    The warning regarding the constant value kernel hitting the upper bound suggests that the signal variance is quite high. Additionally, the use of a `StandardScaler` was critical; without it, features with larger magnitudes (like X4 or X2) would have dominated the distance calculations in the RBF kernel, leading to poor performance.

**Final Conclusion**:
Gaussian Process Regression is an extremely powerful tool for energy analysis modeling. It not only provides highly accurate point predictions but also provides a measure of uncertainty (the standard deviation), which is invaluable for architectural simulations where understanding the range of possible energy outcomes is just as important as the mean prediction.

# Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand'  using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

In [ ]:
import kagglehub

# Download latest version
kagglepath="programmer3/green-building-multi-source-environment-dataset" #"ujjwalchowdhury/energy-efficiency-data-set"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

print(f"Listing contents of: {path}")
!ls {path}

# Load the dataset
df_green = pd.read_csv(path + "/green_building_dataset.csv")

# Display basic info and first few rows
display(df_green.info())
display(df_green.head())

# **Answers-Linear Regression**

### Step 1: Feature Selection via Correlation Analysis
To justify our choice of parameters, we calculate the Pearson correlation coefficient between all features and the target variable `predicted_energy_demand`.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate correlation matrix
corr_matrix = df_green.corr()

# Visualize correlation with the target
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix[['predicted_energy_demand']].sort_values(by='predicted_energy_demand', ascending=False),
            annot=True, cmap='coolwarm')
plt.title("Correlation with Predicted Energy Demand")
plt.show()

# Select features with absolute correlation > 0.5 (or top contributors)
# Based on common energy factors: electricity_consumption, equipment_load, occupancy, and temperatures
selected_features = ['electricity_consumption', 'equipment_load', 'occupancy', 'indoor_temperature', 'outdoor_temperature']
print(f"Selected Parameters: {selected_features}")

### Step 2: Data Preparation and Splitting
We isolate our features ($X$) and target ($y$), then split the data into training and testing sets.

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

X_lr = df_green[selected_features]
y_lr = df_green['predicted_energy_demand']

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(X_lr, y_lr, test_size=0.2, random_state=42)

### Step 3: Model Training and Evaluation
We fit the Linear Regression model and interpret the results.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train_lr)

# Predictions
y_pred_lr = lr_model.predict(X_test_lr)

# Metrics
print(f"R-squared Score: {r2_score(y_test_lr, y_pred_lr):.4f}")
print(f"Mean Squared Error: {mean_squared_error(y_test_lr, y_pred_lr):.4f}")

# Plotting Residuals
plt.figure(figsize=(8, 6))
plt.scatter(y_test_lr, y_pred_lr, alpha=0.6)
plt.plot([y_lr.min(), y_lr.max()], [y_lr.min(), y_lr.max()], 'k--', lw=2)
plt.xlabel('Actual Energy Demand')
plt.ylabel('Predicted Energy Demand')
plt.title('Linear Regression: Actual vs Predicted')
plt.show()

### Step 4: Discussion of Results and Conclusions

**1. Justification of Parameters:**
We initially selected `electricity_consumption`, `equipment_load`, `occupancy`, `indoor_temperature`, and `outdoor_temperature` based on common building physics. However, the correlation heatmap revealed that `ventilation_rate` (0.73) actually had the strongest positive correlation with `predicted_energy_demand`. Our selected features generally showed weaker individual correlations (e.g., `electricity_consumption` at 0.4).

**2. Model Performance:**
*   **R-squared Score (0.1754):** This low value indicates that only about 17.5% of the variance in energy demand is explained by our linear model.
*   **Visualization:** The 'Actual vs Predicted' plot shows a high level of dispersion. The model fails to capture the extreme highs and lows of energy demand, tending to predict values closer to the mean.

**3. Conclusions:**
*   **Non-Linearity:** The poor performance of the linear model suggests that building energy demand is likely governed by non-linear interactions between environment variables (e.g., the combined effect of humidity and temperature on cooling needs).
*   **Feature Complexity:** Parameters like `ventilation_rate` are better predictors in this dataset than simple temperature readings.
*   **Recommendation:** To improve prediction, one should explore non-linear models like Random Forests, Gradient Boosting, or the Gaussian Process Regression we used in the previous task, which are better suited for complex multi-source building data.